# CardioSentry — external STEMI validation

**Nothing is trained here.** Five existing checkpoints are scored, read-only, on
a test corpus they have never seen, to answer one question the training corpus
could not:

> Did the STEMI head learn ST elevation, or did it learn the Chongqing scanner?

In the training corpus every STEMI positive came from a single database and no
other source contributed one, so both hypotheses predicted identical outputs.
This corpus contains STEMI positives from sources that never supplied a STEMI
label, which makes them predict **opposite** things.

### How to read the result

| STEMI | AF / LVH / NORMAL | Conclusion |
|---|---|---|
| collapses | hold | **confound confirmed** — the failure is specific to the confounded head |
| collapses | collapse too | **inconclusive** — generic domain shift, says nothing either way |
| holds | hold | **transfers** — the confound was in the data, the model did not use it |

AF, LVH and NORMAL are the control: they had multi-source training, so they
carry no confound and ride on the identical sheets. The STEMI row is not
interpretable without them.

### What you need

1. `external_testset.tar` in Drive (from `build/04_package.py`)
2. the training run folder in Drive, holding `<model>/weights/best.pt`

## 1 · Runtime

GPU is optional — 376 sheets score on CPU in a few minutes. `Runtime ▸ Change runtime type ▸ T4` if one is free.

In [ ]:
import torch, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__)
print("cuda    ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(cpu)")

## 2 · Unpack the eval code

Carried inline — no upload, no clone.

In [ ]:
import base64, io, tarfile, os, pathlib

PAYLOAD = "H4sIAFDUhmoC/+1923LbSJZgPfMrcuCYFagmoZtlu+hW9bAkuqxoWfJKquruUSkokARJtECABYCS2SpF7Nu8T+y/7Pt+ynzJnlsmEiQoyzXumZgdoi4iwcRB5smT554nvS1v658++p/eB/4gSL/5u1zbfK36u72997L4jPd3tnd3dr9Rn775D7hmWe6n8Ppv/nteu2/UJA8nwcHO6zevd3e3916/8rZfvX6992av9s36+v/+Cu78aObngTed/93egYv61Sta4zuv97ftv3C93N/d2flmZ3/35c7+/varPVz/+3v7L79R2/+R6z8K40GeBfNPd2FY1e5zv/8XvV78w9YsS7d6YbwVxHdqOs/HSbxXcxyndtFP0kDl40ANwzv4kPphHAzUoZ8OwuQiiPN0rvrjoH87TcI4z1QSU+PgUx6ksR+pPMhyBTCms8yr1RRcDF1ZNKeazfsgHI3h8a1+EucAdWuQwuu2PsyP6G+fXpfR67rpLM66dzvqZwJXcTWb+NYsyJXnbQ383N/S3enqH554NpnRc2mQzaI8q9X+9L59qY4v1Ped49MfVPvij52jWnP5ql2OBT1hPJIRqwnIU0BJoH6ZwYtDGPYs9uPsPkj9XhR4qnMXAP4uLjsfjtU0ycIccdz3J0FtmCYTwuQhIGv0C8JsH14o7LsfD1ScqAR+TVWWzNI+PANoS8PeLIe5gfc14L66D/NxiNPh5zXpjzMI8qAPeL64VEEEU4B9cgii+an80qzvx3GQOiqcTKM5zKvfz6N5DRtl0E81TYNB2EcwmacQBdM0yZN8Pg02gBjuY5UG0yTNofE8g061CPw0SJvScWjeC2ppkM/SOFOAHE1BjJQxaCSqF/T9WRbQSPSAx36GSOBWcTDyEXUZjqQWEFJL6Cm11ohGgrwch5merCyY+ilQJKFg4qnjnNAKM5otPqloehg4Y2ySDIIoq8X4chjsvfLlmcjvBRG1p0nBtvl9osbzKfYwC7Bj9xqN6uzjx7OL48uOwqkbZS1eMlHgp3FemjQk1e8ULEqgU0TZyY9H39M8frz8vvnnE3l5Ng4CoGELBk0cz6kSGDH80szCCFaXIB+6WuAc5siHCVOAqPsUyAKQ9r6jDs9OL8/PTtQlro4P7T92LhTePu9c/HhyCX/aR+3vTzpVC2XFVWtLn/tJFPnTDNeNCnMmIqCSO8IUocVT7XhuZm0cDnPVVINwOAxSGEONmcUo9afjRnEbUBUBE4DH7ZvTZDqLGKEAIoBpyHBKmISQ+Go+IqUPKwqRSxQYUkdggoE6hsksHjSgn3h3kEyAWLhHQFvtdw118tN7evD07PxD+0T5wktpuSYRrpiEkJwGSKQDNQGeE+q1oblJzZVJRUiHY386gWXSA1xYy175UcRw6prOAEV+CsMAwmeAtaLDPBhokobEoahX8DHOwz5wbaYbhJfMRuOFHwFxYCUEqQerB9Ax4oWn+kBhMdJ2szdvwh+hXZ7UIXQva6j2uy3AyJZgY5xEA+G632m0UP8QmWkAr8JuZtOgHw7DvreKZVdel2UWYlM/4YKh0TwTTXEPVZ6MAmIctkj4To0CWC9hvzzD6gJ5mtDkF3XO76GQKY04COm19/6cJueLwLHAhTUKKnxMlBjGALgfzTKYG8+aBsR4thqQTANhjIbsRyEsB2bsuqtf1LXgU5ihTAqZwlAYq56MnVjmF0EbhANaejDYKME1h8uMZWg8m/QAfzny86yfhlNY2ylpJIQOYO7pXQhrKhkiZQ1wxmkx5yyIT2UWYQ5QAMPyAooHTSQFjoD9N+KY8ZgCy/YjELQoQt4D03t/dnJ0odrnHXXa+alzDgywefnjabWmYPG8jt8fW+oTLdhQJAoKSZoJXNUZzxx0429BzL1B3nh2fvzD8Wn7pAbKVDhgPpZNozDXKxy5CzKde1qpqDxM4edg4KnzoJnPSFeh1W/EYO0+mUUoc5X1MypxTeRy/OMYgfs8gYzG1I9vs5LQUT14L2gKQOuo08AjgH2/GAw/B5xsNgSVYoZiDP6DtU4zAvMVk6wNc14Q8HLp2QA6kyZGFZFlNAHZHWYJThx2GhUvJFt5WXOIDHYSALPsZ8pt/3h+dtiotX/8eH5YJ/ywlgITjbM6ypApDpNUGTUHaAaAtzRbjZEKe3g768/gqd4c+lhLYMpgDuD1NJseKdA0V93ucIZj7HZRj0J9iEDQhIGKqe+lIxhFFujvf4VX6s9Jpj9l84yBTv18HIU9DfEjfDWgYDmAXo1az1TfmgIW4Qb8Ox3Uau87QKsH9JALvQPp3+3WPRhPEt0Fbt2DjqAsPe98PINm2FpuZVc717XaC7Wg7gLCwszir7jQ8nQGMorwCPSRwYcJyACmGuAmMI0z0hwBGs4wTw/M7xCAikZ5HqDeGSBfw/f4IPcRMKgoQooTEjqWxQHAmBdkNLE4UUwdRFaoE1Nz3YYmFBsy3ZBA9Yll+SPU/GAWL8/bx6fdw7Mjg7Ak88BOCkEb8mBNuc5h+/zo+OyiAyrRX6ih0/g8XyPUbinHKMxNxKKDt6rMHader4VDGo9r9Qhas+oJdpRT94jfZm69Re8HUsE7uTt0ylOFUFjuwBp8KMA9/hw75Z47imyOpRGCpGSrEFYX8F1cZLTudF+gY7LcqGM17AoSrAc4DdLc3QYtJU+tkcDwYOpQyIQjfL6gCAKHUzSdpQFT9laWD4D0PehF2gctj/7chRkQDXwJJxOSoS9Y27ZHDkYJyiCAuFV0FWHzIoFpP2n/8/HJX4D5ExMQdR1ADWcxmznMuOIgGIiqLFp9PxnjMouSUdhvsIBjgp5GQGtMvdinmyaw9Wh6QxrbfZLeIocFZgqiAExrplBQ2QZBMG2S2kJmWO73b7FTOTwFb24CRw/7YwAHq24M0igLoiGyaRxP8ClI+yi4ma/j4gbyHqIhDyycLBTgZyB2MnWYgH3iaRbB2EcWcVhQAKyn5Be/pTovt3d1Q81JoeWH6oa1k/b3nZMLWDGHHn+sfX92dnkBE/4Rbu5ub2/XcMIPvsoFWCeZXgPmQZ+7Ieion1wx91u0bOuo30wH3hEQwLsUuDovknDwCTqkHQOwnuhRr5/d8VKQRQfNFlbXwgqbhFmGU/UALR9hgmDdgqoTRoOt7b3uJEhHAa8EUmSG8EroCqoiXXiTC8/oX64cv5d1ca0419AKb4AglRvexJ+6kT/pDXw1bdESKno+rTMM5Lgwraj2MOqL/gI0+EXg4icPmB3wHtcZAtryvV3pIBvk0AgmCXFKOor2nriDYauESMLsKahD/CIQeygdslkP+wX8GqR8OETiZDCoWI1SkOxi0echrARWTICYQQMTT9El2MnwNCo69CBzdFi1YQ8FLWgXapAm0ymsB/g6JvUPpEEGYjUbgpIjFohSm8phiw71tsksy8UygpfDCstyctEY/wtZ7A7MIK5L46hCOaWxKKYn/Airsam9D6z4FOYh4A20GW0cssYL7wKrejgr9N5bIDccpYGSkw4agvDb/9YIOxw43Yc+ARuEvgZeMThj+7Hyl6E7AaUHDsJS7agbqMqxB4ntHuCBGt+dwhqiiUHJZFxCpGHinEGXfFCxiZtk6jaYirKJOKRpgtsErn101DkSLdvmw/AWZNTaKUOcFEi2hV/CiT8CPPlz1K4GMxhNHyZ6wOIUlhfpkKCsBXhXEM6KADswkCSSGUp6H/WjEG4By2fvoqeJk92QaQLNJhkshatr8UwGaTcNYGQDWh/eCKzfaW/uOnwTxGsW/g2UI80WXOuBfzhQO3UP9BObO+h3eEDcYDQDk3gA2neXn8tmE7def1R8083qgJw7pBfyFwCdOSv1iaGDiCdMvLVoldCaGSV5B5XWJCb7nXGMLp3O4Q+w3s3YBWELg5e73XABATA7AWLPevLKhvKdAkVRMAVT41J7Czkv0OgCsxskGpAp+a/CjKhIUIM9BB7yVyBB27MFki4gSazpTGQPiEoVxOSvaJKNaSigcJOQKEVWUPg4Qk0ewcCCphthVCwjEQ+rGAkW1B6gTFQ10G60ZhondgjaEqyroKUeihE/6m7gtFoOm2JyzQQuTfPQcSf+J0VUQ9CA/39CSqm/tT08QGcolgEnwAnmDJkHZaYX+GO3n0RE7hFLiEI+0BQhKTrsmXF44sjFRJLCRSmkfwOZDTSr/ofCuwbwNRGxDxLyAH78Tm3XS7OPsJ5eGabVI7JpRBYyPe04Q1U1gmkQxk0DSkDXmusRavEL45vS+KY4vgVxKrKclPhpoS2bngqQz/ZT2kFXaTFhZ+VWQwXeyFMP8vWqtXf9qLvIRlNGtoz4hIkzzmISOajXsW8NRpdlLKiwu8QLsll6B+KBlE6BxkqFOkYRib6RBkqMMdgOtkM8DZhZD1BZE2wWfmztanohLk/tbENGD6rgjJk8+X09jSSHetslZXEQAEt2GNMekMFsEmeWdqS5yRME5MRd3Z0uAc64gaYfi4YQnEVDT8wPtaygo4Md1q//FqSJhYXmE/yVlyFOE89Nk+bGTCLoD7c0KewpLat6GmO6l0XfQRZmgTqfxchgOmmapK7T+fNl5/wUpueyc3GpLjqX6vj0svPD+fHlX9Th+87hH9W79vFJ56j1c6w+1+XfKUdaeX9NwtjVPUAr6+to3TDn5EPvB6Qg0jeXbKqGWtQP4U6A3jcw/sN+mnR7ft4ft9DBRppjPPXiAaxrf270x4twNEnCAWHO74Wg1ITIh6/AvHt5jXEz5M3GqQAcFNheE2aFXYwe6d3dooGnpb7YL2Qv2jfIMQkL5KganS+WrEg0MoL0rYhZy/7kSWc/A9714qQ7Sv2BrRcMkA0feSB+L5DtI55YrW7wew7eAa8L6qY9vwytFIIIcgwMVxooPnpCv7olchgAsgjLXZTYBxbaQZiNZ8NhFPBLGmhQd9EMBa3sYLfst5iGMZAziJP5Ab+5Pxv4Xph1/Ts/jNCh4taLXtLke6iCuMVNdHSThmV835OpGUgFOI5xeO0PH80TyMyJzcKYuvAvchvGSJkXcBsAzh+8PHE12YHq0+1FSR+U7dHBZTqzcLswW/4sT/pgErgOds4Bdh5jxwYH0O8F3sMzM0J9+4AH7/Kby8ABA5o78Ssypm2Xn/XI8AI8ev3pzK175N/QaBUzDBYI+vNB543hPxcggsqNkna7LvYZ2byFI9lFecdWLwweTV00FfUylHGsWgvsGCHb/oypGVDXM5OGryL4sJr9aRewSgrngQP9B3xJJL+LS9Km4xdolEj6AN/nWPAxouwU1XhJAUBuLQES0rUSEEz3YC3C4BuWvBok97FY/WwikReF/BloeiNfVX9FCQpQBmGGLjXxG6J7D6RrGIMWlrFUE74FTCyBcX44YwbixsS7FvvN48lymApojLi5Yh8cKxjABdBNA4zRxd8w2tjPma6lGRIwoRRtL/pUdILQ2yXgXXzQpY91q4Eha5EvIHqkG+yMLMIFTv2J/tC70Wa3nRzwrK2gXxibKhj6GJlEXCdq29vX3lcQfmxhUt82MttvbwHCeNSyD5ZC9NlGg31ZpGgkaSZKcZaDZR9qtaOQmT+BhSMSs1aW0Q9IlI8mhgLWpIRLCoxQJGsOpJZSeIOSJagrzgKsJA1HIeavLAZW2C+pNTMKATSBE5Fmhgb62yVQOFwJ900sXFlRBMYNADCYSdFZUnbFCIlGoLG6MKA6U2uDyeeredLEi5mx04c+dyd+dptV+3yQlK6IsxQy/LrsBAILXgJakgkCCMDVlZP+K56HUwBYduJonMr8DKN5kZOgjTPtHAElBJ3aPoHqRyGbuX46mlE81ATuzR0gBolxoQOOiDJkM2wa+To2rFWGLO2LG44jG841LMGusGlOrQALA5o8RC3yqtm/L1paj7xkg0+iFxtCM7qvNwyjKPZdB63tPAV+ANQKcmHhnaQ1C4wKDdpuzrwDJxE7aajTaZ+cOC2cOBh45pLBOgSyGqA/8KCXJFG9YS1hnMwRxu2YKlroBOJID2hdMY4QFlw/AuW4wXTMwUD4HqQcAbaAwToHHag3t+iCdCuKe6BczPJi3TvoG4Ce0kQcyFe7ax/T5NN8sUMYD4ox/k2UM0OPPTq+MZdDGXOL+mi96ePl938+6X44tt5mbjUWm6FJsdSQbi41NahGZuzCE6DKWpBLD18L2h+10UidxMCFzloJ/chkcAAt++kUnXrk2ZM4uyTcqY6fRnMBg2iOMEjqi/NEfd85Pf7htJxW1EQ0ki8i55BxPEM7Z45r5j6zjM/iEd/EfaFFg+Ww9rzcjxPtEAV25af9nPN3rOD/CxOQh46Arjllu56cgBx2hQVK/sAh0CSGruVp8xzlJwmsgqty6xWZTNimSHKj2556DwqIgLHtYPHHsmOWoiinZ5cKFcWJTxkx0LtRkqDVjeOXibGwnhrXUQZjyhImOOawnF/mbqIdEQUZ6WmVRlFZHaQf3Cv0N9Lay4nR8DJkSMx6mCJgLVyXFrbFE66cD8cfjg+7AZJKl4iE4g6maw79Ysjnb5wzWF+GQC26fi8Gi2sBBOjfTY3SZ0ACdQ34euh3UW1bAKV/w1VDP8Nf0G/glqMVzXYUWR4QYf0Y9IwxtwdwA0om9dKPWAcdIwcEIzCIE1ALfdBzEXkCzRkkACXMKdeOYwK2kzxKEmCss/geOMofHHss79vnR93Tzg80ABeFxJVDK0U7ONADInwcv9rPUjt8mFgHAlh+vqQd4GNfzbQHGsmBDv0pqQH4rdsP3XnLIsoGhp/sr0PQzmOy6aGzJtRXtieBMQ90k0PvotM5IlLPZ9MouCI7CODgn0KH+BgARweWByL68LiBihEYYhH5cdl/S7IGSAoFNwp30Sj+hLfJomCpVTh7iA9Qtg2m5E5Qi0eGwFSOKS/oMeBQDGcgEjywNsgNrMMmxnlHjmsr00x0CxI2PUz8TXLR7EQJZH0bc4vIiYE0BApggAkz/VuGSxyH820wNIYOZ3TO3YcDOw8RvuZiuGU2GQ/DeGAUZ+P4YBcazBgM8ZdZ4M7rdfV7tbvEXNgcdWIfVmaj9I3pjVysACaFfoD+JpZBF+67OL3cCkaWFWY/MiKy2uGZUeDGlh3NwVd42KOgIJgDmBmAPZ3X9d/6oh/QGsQVALheHAmH6NBBNwvMzTydl1tgH43bkIY5jAUgELcALt4dfOoH09wyP554n1hT+IYvxq/dCgY6NfTvIriG2vX26+bBqgbfvsYW4hdAkiRrMltavmnSK99B40/JKkR6hTXfUiQutKrPg+Hc51vKTfB0JoBOYULNfBR0wWbukzusS85nWLhJv+vP+vy1VniGyMb94PWA7abzrkBzpX/UJ5kDaA6aLrFCdHDYpME/ISM1P88lpGb/DiLB/O7OmecuNTO2IjVlLGMnihaYcdTF3pkXciu8dUVgd649bARwkRKkK2xwL833Akjdx0WQ2yWQ5d6vgAzt9CxajsSGGofqoODpGtGl+Sm5764cf4a/RokDC8O6MQ5ZMBHMZ7xiBW0svWyaLrwMbyy/TFYKtPmKiSSAVszwpbWDH1yifaAYxqGPvkudpue1xZ78iN9SdxBw4ik6w7rdAWCo+4w0MJY2IA26ZDwdGOjn/v1RAfF9EE3f6aZ16YznDwZdbdW6jtlG46Cg/GUG+sqAvJ3V3cAspAMH01TSBKNMksL1e1Kuv9sSWFs9yirKn4o3OC4Fvb5g807dWTkGSTJxGtrrdIBZLlYGIubooPObUuUWd/Y49ZWAgVKeBipbfp4Awa5RB32q6Sg7cDYtgOhLewrRVjoMWS/oXMyUK4+3SrnomJr3BIrY/7fwbmkN/UIanXpERfhY5pr441EAFiR6msAWISXDdrlAt2YRRQunwUJSm5Weia4WgcapvSBd5pr/G/9eMuMMmKUctrcUpKEMTFCMQLXBNDuBpxPiw9FWWayIl/b5DmshhoYIGYr3IiY8+QEEaHEP2jDu0LHOGZlmT5NT9ya3gzB1JfmVlxOntXeTW4klFDldy7lnDHoxf6pecsosONr4x3k3B+Ds4mHnke3V0XlbGEOT9xNRGDc9f+URyk+ghklYA4XD6hAMyRNy58tK0NkNDKelHvjDI6r0pIXrhIdhHe/ZnGLIJsvvWpzEwIO6ajXEH+YxrjbYx1G/NpkwZRguadk2BLaTNvD+xvVnoZEJsLUAVPJJKqBqz8wzIE/R96SXqsaT5FGCIj7CHQQoPtTDoWd9d+smK4CYr02jwnjFz0dcgoRO5gmhwzxeTUirnqBW/eHM+3B21DnpnrY/dC6elDioO9D7tuBRwIeWF/hZ+LxTkRaByix1pDLxMU5WyowZZTM84CuXU4t5FhRZ0DoF7GGjoTY4ZF0eFuHLxjHjovQEdbGOL9KoZYbesvzU+AkViYdHY5rgY+Q4KY+wfzvFeRF0UaMVGFvKAmo2m+oBn9CdtmJMRTDC9uBr1mGF8PD9HLwzcbuKdCNlbx8Jpkl/rB5MLGiDbiCC/rCBK3M5vcjaW/JwlaLYceGle7bzyLS4BgiuhFTAMrZDJHWNcZ2ToQ4WMwKKHACYWCseTbNm2VkoGss7g57iVWUjzGoWTKb5vNvHLGe31Ldg0CVODVz1Sif2oSuJffvwiZkxflrKc1mlzjnocQa239VZAvi08eyH9AKaMng5fLu+9kDGzq0oOWL7tqGTdgPg8ZTg6jLzWcp5oTFcDZ159wEeeSTNuOCst9crmk+t5mRaPNkYPxXt3eIB9d2BRRVXt9dlYbQACmUWZjhXyFb4PnR4nVC6NS4I4LI6ylqY3rTLHNasHdtslVaSQ5QON4s4KN+pV8yZg5HISdadOC1agtxe3+zidiQ/QutcaAFf9vD4WJqtPi9MFBg4ZSQ4vDAPJtkiUWJaGfwqthrabctpBEveClaIDo8lteWelDXK+NMeIXR+v6U8eu0UAznxSdK4uN8L0D5Z21rRZSQRcNRIaWcFBtrpN4rL3odZ4C1lSYBBh/qKZpkPFERq6JiMqohpYIPHMo2BrUPvMVz4Ny0DG5ROarfdHYVER6JlI7T4+iX7DkvEbnwj5SyPiQ8cbcWIYNZgOC6bzYgZMmnxw3CHeE8RM6evRezBqRiz9q0VY7++gldcL4Yaa08oASTQMRoV+7G7CKh+vfQojQ6YAv3tPkCrR8tNAZDIOYE9YwcF9bHaLVFa1FdmhV1fEVGRaHZiWpi5W6wbXIzYT87Nb5mJf/Y8bm5S3x8tdxzpBlf6rdSfgudk+pbVRfMjqZMHKmPGIbT/8FgXRmL6ad1kxz3dsAXlp3kBx1o7vwVWoRGwN5p6SYpznTIxQcLjHVYMLJLb0M7EDZikjXq95e0NK3UF2jVpAyF6fvbjtFNv6k6TrG4DKbnTKoE5T46SkOgS7uoyTLr128dZ6qgF64t62s+jYmaZSa6aLikGQMwLN6eenNRbysEUTqXTNxczXaJHJbMB77naMFSycY2LWCbmmvu0xI8i5kbtd8S4f3qPfyQ113Z4i52i88fnqJ+WEwjYzOlaZg2szAUzxwqFS25CS/iT9Yv23ZQ2NNmP6qQyVMalFX60m8RdNkWFdYg5arcQKxsTIkmi33LLO+EwWvrcLcnzRwuIiSMJBNSQGI4InDvQYq5tkOFqgfZYW+kT/HxPTJCua+Q/9Kki8iamEvwoPE/nF9g+D+NH03sucfMx2IKYeRd0MYLs4h1vMJtMM1fIgRW2OD/YrRtj9n4h8MN6EqtwGALiLiyrSpZS1WdklZlvtXIlW90abAn3ryyGueIJ3Usd+nmQpLyW9FTr/y3dG0fLHHoPyBFZxnYylIsQ6wuq7hJKV+m4mhnc434r9cCPb1TOyMaiGcwDWvkMvPIzj1gK+cbjlhjy1FX7KefnGMvUpLO4ZSo4cWpDsalRHPK4rxNUgG4X0dftUm5Mt4vO9G7X4bngTL6LOYbtO+hEYFd7fV1s7j/n8tb1H9f1H4v6jzvb+2+8b7df7r7Z3l0vyf8Gl+Hkf8d3fKb+49726z1d/xEIcA/rP26/2l3Xf/xPrv94iSJdlLatSpUEld+Ed2VTqFCNZ1ifDIso+X3cWl+u+2hoTa3aatX8zrxQb4nr/tQ+OT5qXx6fnXaxbsr5pTcZ1KjkYpFVnqQjPwbFYqB88mSrs9OOGkiuARd64uwp2eoou4g52qHypJbNplxrJ1W/zPwoHILGmpfKmOnadBluWYiC6npm0H9T1otMpHQSDHQRI3vT8vMvkzspVVeKfLPFMmZFD6tKmVHn7CJmb0ulwp5fKezJemE8MdAnKmlI5QcmUyxN2SDHk2zmwfRvLCSld1dL5u4gxXp+/r0/Nzl2NV2sh8JJceDnYyz9pW4oYwKmqQu6LObQ3SjQ1EFNxZ0k2mNaKp+VpDXHYEjqXrI7WUpLAETQh98WZYFK1SbsTQK0Fan0K5Y0wqS+nIqufcXaTxOs7LS65NMzSzmZgixXxoFU6QO4runajti0uoXUf1qs9TkOogHVT6VcdMruyRrkqaa5rL0onmjqJV6VHaLNkC3LjFEu7b85QNB1D0D9T86slEQFU3BW0wpvmgaikFzdhCr8qXfnZx8oyZ3JnL0oyRDgbXvfvuG8TN5Fj8SBO500ZJx/zrO8B5IJdJqltbCJuqNghKUBAKDOg6Bt9GM/wuRTKioEhI9/YRCCR8NccPs0J2boSpEwF+/OfjzFShqY3SkbmPSvk8DPgLRoUwtuLaIKSlTqw3A13KCU+7dBQ6qSgKDDOnC8t26CEbhg4NVwpzHtPdYOHkyGiYN8fxsMX7kexP3LbuwWIuz16210WGP2tX37zasd6zb6uun2/qttcWCg6+guBsx2e34W0CsqoL/69mUl9J39KugvX+1p6Hej0c6rouMroL/arYK+/+1uFXTQVQz0ELrd3Xvz0ryhCvpeNfT9NxXQ37za/1ZDD4bo+ocJBex3e3voZlqGvvf6VRX0vddVmHmzs4vQH3HV/smwRWHPuJXZj0czLAnCaci0LUtd5FRhxc+4UpuPlRT1TgqflnSQIo2PaXelP0qDICv2r+H+Hs7R1hutKDMId0AEgSlKci9FZgazPm4Qyj3gPCcn7Y8XnS6vywPo/at9UQt8EszM4GkbnW8iWUjZXFYHNyr4MW7zrWFJRgvOm31LxWBYnL9kw8JtGljHFFPBefc4pUtJMu3Q/QSraHDwkpICs1xSgMOh+oS9KAogWd4Q59/+1/9mF2wpAfnOBE+k7pOkF7uXwBspvbhhpRrXl8CiB/RTye3ixFuc1zOhKmsU2bnTuaHOw13Le4gHj+gPlopOoTtpYGyqPBiTvjkh1zU8icEezMesNxbujcMi0zRKNAYQsQDA+uZaO1ARPI2bN8RafY2S+vIwhy6/ErtZzo92Hkq/PaoruIHQ9+qPDQWfxyF9vjbjXdAVXGv1mLRnccWX7lpbFCiDBP5X3uJYlLxljcWqbI67tpI+PMGCgmtv4jJA1PB2XitB38KG1bllrDgoD4Bffzg+7RyhXF6ol92nRSfboKT4R1Hkk7yb+pWlAavfH6jyClx+9fEpSKTDkx8vjn/qwKvLkQnHqm0s4YxhgLWpksQqk1eUbYZ+cYaQFZguxyscKb/8Viq7WhV3rSq6Wn4u1dItCNRC6HOGqcUuyd/j8w+E54XBlk2CgZgEJJbtKs6MB1SNFPAC3s3qh5GUVlkY7qKij3RCaLmnUq7FLhYsEs/k5mNdrFGkC29WD/m7A1Xww+XhXp63Ty/edc4vVo2S67saJcsqIEM81Ko3kyuqhCKFWSOY2WhxkMtUMkjeLpRDXlk3eBlnnE6rawNj0VneuFN2Szsf2+eXx+2T0ghldIMAK3TgFAaUXDFPZHNluZdaoZMpRxrEGXVKEbEMO447H1EW6hq5M2pMimB5kyMVuJYS/jMwSfS+jS/NPV+dscwa9PMSn58IBK16QYL85wtyq7/U+IVHn3QCmJ49kfc8LaVXyrjqdnLjtKK041KQwirxOH2kmcfcefscCzD1M012RLIHZMdRTYfMnXK5R4ql1a2KDrSNG1pf6UDdtVS7+xtaT7LNm34vRTBxRyfdLUKS17ViZylQstSv4KcavOuZRf4SB+A8PcmGsKNuKGE1ADuAra/VaQryOv0VXlve7lSvF71lBVdWW6nbVld14kvV0CipBvVga8e9mLHXSwDuqA1FV+mWEEFJcbpe0rhmE05wAZLE4DJ/Xp3rQs+ftKhgAyoO10VYtA0fTyT+yO3arvOifKYKkldHs1vmUj+ZUhRCYm0sEqA/bW4eJbK70VIF+FCP0hbvBupkek+r/F4+80MvUkcOivjD5qaqOt2k0DNykvLIqGWPep+qb0s9P4DiVXX5VwX/WF+bzSb9Z24NocnmJtVIyqAPv6oHovgNHevfuH5U//f/6HdUlTdUi8B+QB8dIYgBwuLRWeRXkt0NQDnfHOdA1y9ogKZCdQWYW9eXAFMRgiXIRSa5DV0f59DHOoAwVfgiC3jDzMDQ4aoF5YIFyy9/Xyr6xu8nLsGZK3on8kZDbYOqrDc/6y20VFMP5zVOrDeLo4D05qUXHi5luNNLb2SCljNDYNw3S1C+1ykM9uRW5DUg0syXhprae4Ltvlm0BXRUui1bTJr/zkubzWbVvlA/8Z2qTpxz+X50G5H9i+Vne1SF1lvy+/B5AlTPvFBxkTDYtWTWJOWDmqMJCKxUKi9pLAKU99coPBCgsN71YUBmmdMRKlWnnyhRv584/GTFyv5Aes+vJTcbjUYcbMrVAIJBvaKdUTUp1+5XdZQmU/hzKMPjhm7J/97gUv1M1AjzXOpDVPCYVsX/fi22ZLInpHKXgOw7KCUKWhLJZMIaH6vIJateiR6EeO8G5vwespOwuIRUvWFHoy7mcVk66MCCRyTBhGNqE4TDkt8RB6SS4bA40UBv3MEvFqzS4Tl8wpM5Iqehz0JAm81TeDQYl+3AqbPOILLAWSRJxxRZkQquIBUFtFHMiA1K4+FNZxM/+2UWoGJuAcQTDOzmvSC/R85JGnqRr4wle6qUCp6eYjJkOWNVuYMKFwFoGKXtqUQXJmXU5gVmUz/S7IHS/lTiv/xqk75pu+2sfQ9I4AfKNUCaivWL4o6sPfKrSAnnks+gvljVq+C1nGWPTHZo4NXle6Y/YBf05z592Nx8kEE+IouuWupIlZsa5KbmO+IpN6eyWLGC4qwN64Anw4sWQghAHcgNOJoTVJ2Wdk8UYR1YRsVrKLKjYfJzi034wTI0OvRL+i7lT+362QYiPstFrJ7pYF/C3FMspYsV0eZVBPkcRvOclOgnlka9RDlNJAGiHZh/nIkH6NmjtyxacwqQlDgmKdh60XhoJgUlA4seubrDlUQfaXXcoYqifqd2jKKwudmGzmLNFtAxufou4QsIVBAHXaN02cZSuixuL3lUD3e6itajwz1rcKmcjOjQ5bdLtiAZSAdSYv/2rqWat3dXO1hjAt7g/f1Vjar4LisaO97CqVba/kGdQvx6lQsUC8OLRdeQOvHFCWj6ZMKsamxMAEgYgGN3hbJcOIO2ROvUjR6NMCZV2tI4vqxy2Kp6YQZeqXBYUaoMnc4D+IljdEM/RQmrsgnVy7dqSVNBgsBiGFIYxlPt6B5djikrcoGp29KQcmKmmIy3gLMFFb9AX6VFYFCoMWUjqOhWUdKMHGPaHEPH2Obmyvpmm5tFySygbTnTyUDlmlgSfacaR1zH0F+oslV2XKFNR4jm8w0phFKAJFvIvZmEN7BqLnIAMJuo44YKPcAp4aQu7DIu5vnislkUFSOwK5DK1guhVMitKJ+2SHTmWAWjhZDT1HTV6ENDfxJGoZ8aG6woSol6720QTDM9LQSCDwzS594tjF0fHiXYKQ6T0XSRTj6jOB/y0v4VONWv5AT6VSu93+7/I9g9dbrx8bx04wL3P/yTVfcS702DPvyxdzWYb4CdeqUJvkJFXvrfM6QZJXbzYGjfwdNbxRpcl3AhaXuCmtFvdVGpz26lqddrFRukJs/cp7eoYXFf6OPkaiNGipSPtHGEv3L4TXZqkK61vBGlaDRNpRFGva5Ku1quKcxV/FRsHit+qgBNjcs7WhYglYrSmB+riPY7WJo3ejpvpFLgTbzl39DkM9kiByZ6LZ+kCZpVih4oKURhrUs8hC9NboPYg5XOh/zJASuxrpo1lsyvRTVOB5vKIRZHW9Wos/nxLVrW2nTGLpG9ZE42Skl5GwdFjT+0VYd0+rHt9S84yCDgHKj+XJ9tJ2veTpMxyvziGbgFQHE43oBOf+OREQv4taYc0IHd3dy0ljTcSwQtaXLPKJ7ZIs3MEKyuG2Sd2DGUsukED6jKSBPf2d+noeiOiZtIs8466uEWq5snM9CiscKqDrrrIiHEIJnVeSspxur+jVa6xXwgrZqiXugUK4aOzbB2P5KE6caF9fsgQKdqxkUoi7MG7XMTKaxWUd63UE4wz6iQf5jISORSEf0S6pVsIT7ISJcJtMguDkytWu2FCfl4Velfhl85lMbGSlH3HnMQC1hoD4xiLIL5gbBnNSuOL94UYY65FmG+ycMGCmKXL2oLhSGDPZYz8ibQVp9IS4fy2GfnfoKHKeQRlM7xsY0iwARgs5cSOr2v7n3TZGhpxLvmZNIiSldFbhUn8a48bJdGueq83ULhrDh4l9241UfvFu4yTuCyj90tCK/wy5SO3rW8Luh3+ROscq18zlWE02WVKMUNO30LKBOeWLCsdNEJTuWzgIszqjAhK8KD6Ao7hFwTmBpT8Dw2brX/Sh+VWGlBXHACwuam9hChu8pW3sq2QmvBJ99+h7/hDBau6XILmFlsAn8aC8/yXOOP/MkcjbwwkSbCq4vqkvVQMJginkJiwXLbVR6bTHSAc7bsTCvUxFVetbb24JEr57NOMgOw8JZ9xkf2tNrZfify+ldaMfqzrBr91fJgxsZ56yLWULn8wZ8+1+f6fDWyIvqnVcnnRf/6k2f6BL/Yozui0DzAbyoTTHQtx1x/UseYW5WrrnJ7/IJCyVuBf9XOjSGmdZUCmHX1u1JhI/HgTZYcfNDRVTrcuwQPhRSKaRRsR9YFe2AWfdISJ/ud8Mt66+vT1jPI6DcSD1sbz6OdrzMjVcS3Ad3YqK+alTYYkOnIOpsQSU2fswuKgJ/PJJ3TOv+75JzkAG+rrK7ypHIRWcoH5eMt9DSzyMrsDMii3rjJSCp0iOemJukcna+vGzTVJJyA2m7pBnseWb66iDfwauZS3HCFJzsrEq+NHMYTKkW1UpspYa7kCtH6Ff9E3pTNhqWxMd7mrFJpZwpIBklXlQynlEoVD8KUd6VUO4wph6IXzVLaTl64Od2KGt6WGxiU/06phreUgM/mMYj2SSCu3V4QA0mVcgU89YMpAG8ZkQ43IJ8NIGeEZJc9p/C77Z12q8qGl3t9WlU2XDqvg9hsEGH/Ly6bl3TQ+KioQWh3WveDsLvcj3LR8XJHjuU3taXwV/wzo7PKAQWnOuaPfQC+mGAlW0yKBjUhMHau3RE9JLIqNwFFm2Xf14dj00HLDxJjPQkT0ydS2C6VRqaiLp+pgsyetAeioUdQwWLjLXuadS9wvFtPKm5r5KFYdMhFrt59RGM++HymF1E0+4QqyFd9hjw+e1VOK7vxYSB4TB6e3Kq7+7UkDUqYrHziVakG0L9vpNWkqopS81/qNyMy+qpOM9J0rHLL1fWuEEu6AgOlyC8UB8LjoWpVu+BQh7sqlUi6/gz0irx4fLxIjd9BvQ3uLJZweVro00uEmla5ONro5cFDgOW8XG2mkee7+vwI9GxoZz3xjpLdRbVUc/Wzs3B2hfq3f/lXlm8/O7LvSm17WBtMx9y1Z4XMQdspJvsitV/CclZkRgj67NFYOjDjn/EYxaEfZSXDRusBYVZkJsBDIO60VmANjw713ESfibGnqOzpMrxNitjJps0dcYzJ+eycX2PtecQVikjGF0RfWd9QEagQev9eoXC8ZMu8FDAWRwylx82TWaV9DEMB+VxxUgGSAunXhbMhW5WwZoIoC0YrFzjMSi5XUhdBLuDJoLPptNBBMP8KzX9P3K308gWA5uCCUDYUFPtvyRmWTRPaFhfxcQU6oC4H8+ZFgpkjqYe+yaCX0BefoiC5JXSqL503hoHBN6pEXexvK0x52bhDe0yT9FY7czhVglMI7FOCMQIISMNtvkyKkmkPj9pW8i7Oz2UpdMM+Z4r0LG7/83EVi75b8lLeDEbBIIm3+KySblF25UZNZ70ohKWGQPuzlI+2Nqccb8Jq5Fdv6qCUAYr9iFCnxDCm8WDJ2wH1maKwuxys3DTqJR58IsOpcGkN+HxUnnCu2kSOE1ilCJ4QWZyzA5ZHxOZDb64ZgwH6vezQMFHCxZigcnU4kI741RmRcib8YIKOUdawLKhHCcsaPGmVT8mQfWdEsfrcWt6Wri2Qwklsze0ezu05eV4oaSpDnRgAAVtGbkQ+Pwzd0Vq8LLFEMYcM0q1sOkmq1FaQsaVvgv6oSempzdswv1FyMDr9dvTxuBhG2VXkz0jBFoX8UlZxsfW4yBIE/Zj8v2JuAeGQkRhtLgQ9N7N8HtHvbK6w+g9TFeaSR8bn1LGjKyv2dovqzWYiptcWq1l4h7gYpQmQZmR7wbiaECjmVMR4Oqbd1OMk51/tVffSo5xazK5njxcvL326G/HGn96z45wW9f7Lf6Tah5SDpqM6FPMvpCcmgbC2F8bl2dM5duyEgJdNzaubWLAJVQmYg7c6hGVgvtshUjBnGshJQznTs+x/XDznT7Ajqpg16n0ctfhD9GBwjesoOb4AVpnfyyjFWguIgoFYGbzOz47Eb2CiUC3AHZ1ISLSWfX3KOjJun5IF5L0Uc0G0on1anWKhmX5Ee4xpjYGwS8fzfDxpUG8/hXISlo65obIlxGtzr3s8PlnqzzND2siIo6Bup26wSzcEZx4GuIN/d1+LtAqhznIYcPgxTe6CGDccrshpucTkMp7zlklalrpzmKlcavwnLj5XNLSr0S21FvcVILRZVQb8MxnSuBA5Zywr+6kNpeIWeXZjm9ONGxKoM+sTJRJlwaO9SBRpfNQW2gANpjRCS4fIUJlk9Uy0sc3NlWdhSiBSb2s0fZ3FbIUPPHVKgQrMkKPTNjmzZJCwlBLvdo/O4pS0lDwoxRGa6mOQNiUQt1DoOms9nfZ1oB606XNdUQ7bKY5Mk4SyJYJaPD3AnBSAZ/ry3qbnHA+gn7Bq6Tk/x9Lpk4Ua+0UROCy3TfluJ/VHhSEZY6QXdfXKOXZLBe+Kkm9kxrR+v/MG0+Ee1xXb1tf6Wl/ra32tr/W1vtbX+lpf62t9ra/1tb7W1/paX+trfa2v9bW+1tf6Wl/ra32tr/W1vtbX+lpf62t9ra/1tb7+S1//Dwk6cm4AyAAA"

dest = pathlib.Path("/content/eval")
dest.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as t:
    t.extractall(dest)
print("unpacked:", sorted(p.name for p in dest.iterdir()))

## 3 · The training harness

`evaluate.py` imports `config.py`, `models.py`, `metrics.py` and `data.py` from
the training code rather than reimplementing them. That is deliberate: if the
transforms or the metric definitions differed even slightly, these numbers could
not be compared against the internal ones they are meant to be contrasted with.

Point `CARDIOSENTRY_CODE` at that folder. If it is not in Drive, upload the
`cardiosentry_runs_v1` code folder or re-run its `make_notebook.py` once to drop
the modules at `/content/code`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# EDIT THESE THREE if your Drive layout differs.
CODE    = "/content/drive/MyDrive/cardiosentry_code/cardiosentry_runs_v1"
TAR     = "/content/drive/MyDrive/cardiosentry_dataset/external_testset.tar"
WEIGHTS = "/content/drive/MyDrive/cardiosentry_runs_v1"

for label, path in [("code", CODE), ("tar", TAR), ("weights", WEIGHTS)]:
    print(f"{label:8} {'OK  ' if os.path.exists(path) else 'MISSING'} {path}")

In [ ]:
!pip -q install timm
import os
os.environ["CARDIOSENTRY_CODE"] = CODE
!ls "$WEIGHTS"/*/weights/best.pt 2>/dev/null || echo "no best.pt found under $WEIGHTS"

## 4 · Extract the test corpus

To local disk, not Drive — reading 376 JPEGs over the Drive FUSE mount is far slower than the copy.

In [ ]:
import os, tarfile, pandas as pd

os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/external_testset/index.csv"):
    with tarfile.open(TAR) as t:
        t.extractall("/content/data")

TESTSET = "/content/data/external_testset"
df = pd.read_csv(f"{TESTSET}/index.csv")
print(f"{len(df)} sheets")
print(df.groupby("source")[["STEMI", "AF", "LVH", "NORMAL"]].sum())
print()
print("gold STEMI (LUDB, asserted) :", int(df[df.source == "LUDB"].STEMI.sum()))
print("proxy STEMI (PTB-XL acute)  :", int(df[df.source == "PTBXL_MI"].STEMI.sum()))
print("hard negatives              :", int((df.n_positive_labels == 0).sum()))
print("patients seen in training   :", int(df.seen_in_training.sum()), "(must be 0)")

## 5 · Score every checkpoint

Thresholds come from each `best.pt`, frozen on the **original validation split**,
and are applied unchanged. They are never re-tuned here — re-tuning on this
corpus would be tuning on test, and it would hand a model that ranks external
positives barely above chance an operating point that manufactures respectable
sensitivity. AUROC and AUPRC are reported alongside for the same reason: no
threshold can rescue them.

In [ ]:
!cd /content/eval && python evaluate.py \
    --weights "$WEIGHTS" \
    --testset "/content/data/external_testset" \
    --out /content/results

## 6 · Report

In [ ]:
!cd /content/eval && python report.py \
    --metrics /content/results/external_metrics.json \
    -o /content/results/EXTERNAL_VALIDATION_REPORT.md

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/content/results/EXTERNAL_VALIDATION_REPORT.md").read()))

## 7 · Save results back to Drive

Predictions are saved per model so the report can be regenerated, or new cohorts sliced, without re-running inference.

In [ ]:
import shutil, os
OUT = "/content/drive/MyDrive/cardiosentry_external_validation"
os.makedirs(OUT, exist_ok=True)
shutil.copytree("/content/results", OUT, dirs_exist_ok=True)
print("saved ->", OUT)
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        print("   ", os.path.relpath(os.path.join(root, f), OUT))

---

### Before quoting anything

- **Eight gold positives.** Every LUDB STEMI interval is wide. Enough to detect
  a collapse; nowhere near enough to certify a head works.
- **`mean p(pos)` when sensitivity is zero.** A model can rank external
  positives correctly and still fire on none of them, because its frozen
  threshold sits above every probability it assigned. That is a calibration
  failure, not a blindness — and it has a different fix.
- **High specificity on the mimics is not automatically good.** A model that
  learnt "Chongqing paper ⇒ STEMI" gets a perfect score on every mimic in the
  corpus, because none of them are on Chongqing paper. Only meaningful from a
  model that also fired on the true positives.
- **Do not merge the LUDB and PTB-XL STEMI figures.** One is asserted, one is a
  proxy on a source the models already know.